# Lab06.3 – Custom Hardware Accelerator

**Objective:** control a real custom RTL combinational MAC circuit in the FPGA: `Y = A × B + C`.

**Prerequisite:** `lab06.bit` and `lab06.hwh` generated in Vivado, copied to the notebook directory. There is no fake software fallback.

**Step 1 – Load the hardware overlay.** Verify that both files are present, program the FPGA and print the AXI-connected GPIO IP addresses. The currently active `base.bit` will be replaced.

In [ ]:
from pynq import Overlay, MMIO
from pathlib import Path
BIT = Path('lab06.bit')
HWH = Path('lab06.hwh')
if not (BIT.is_file() and HWH.is_file()):
    raise FileNotFoundError('Copy BOTH lab06.bit and lab06.hwh beside this notebook after building in Vivado 2022.2')
acc = Overlay(str(BIT))
for name, info in sorted(acc.ip_dict.items()):
    if 'gpio' in name:
        print(name, hex(info['phys_addr']), info.get('type'))

**Step 2 – Configure AXI registers.** Identify each GPIO by name from HWH metadata. Configure A, B and C as outputs and result Y as input.

In [ ]:
def find_gpio(fragment):
    matches = [(name, info) for name, info in acc.ip_dict.items()
               if fragment in name.lower() and 'axi_gpio' in info.get('type', '').lower()]
    if len(matches) != 1:
        raise RuntimeError('Expected one GPIO containing %r; got %s' % (fragment, [n for n,_ in matches]))
    return MMIO(matches[0][1]['phys_addr'], matches[0][1].get('addr_range', 0x1000))

ab = find_gpio('operands_gpio')
c_gpio = find_gpio('c_gpio')
y_gpio = find_gpio('result_gpio')
DATA1, TRI1, DATA2, TRI2 = 0, 4, 8, 12
ab.write(TRI1, 0)      # A output
ab.write(TRI2, 0)      # B output
c_gpio.write(TRI1, 0) # C output
y_gpio.write(TRI1, 0xffffffff) # Y input
print('AXI GPIO directions configured')

**Step 3 – Define the software interface.** Implement validated range checks, AXI writes and a result read. Define the same mathematical operation in Python for reference.

In [ ]:
def hw_mac(a, b, c):
    if not (0 <= a <= 255 and 0 <= b <= 255 and 0 <= c <= 65535):
        raise ValueError('Expected 8-bit A and B, 16-bit C (unsigned)')
    ab.write(DATA1, int(a))
    ab.write(DATA2, int(b))
    c_gpio.write(DATA1, int(c))
    return y_gpio.read(DATA1)

def sw_mac(a, b, c):
    return int(a) * int(b) + int(c)

**Step 4 – One visible test.** Run `7 × 9 + 10` in the ARM and the FPGA. Display both values and compare.

In [ ]:
a,b,c = 7,9,10
observed = hw_mac(a,b,c)
expected = sw_mac(a,b,c)
print('A={}, B={}, C={}, FPGA Y={}, Python Y={}'.format(a,b,c,observed,expected))
assert observed == expected

**Step 5 – Boundary and randomized tests.** Validate the arithmetic behavior with fixed boundary values and reproducible pseudorandom vectors. This checks functionality, not speed.

In [ ]:
rng = np.random.RandomState(2022) if 'np' in globals() else None
# Python's standard random module is sufficient; no NumPy dependency here.
import random
random.seed(2022)
tests = [(0,0,0),(255,255,65535),(1,255,0),(255,1,65535)]
tests += [(random.randint(0,255), random.randint(0,255), random.randint(0,65535)) for _ in range(100)]
errors = []
for a,b,c in tests:
    x,y = hw_mac(a,b,c), sw_mac(a,b,c)
    if x != y: errors.append((a,b,c,x,y))
print('Passed:', len(tests)-len(errors), '/', len(tests), 'tests')
if errors: print('First mismatches:', errors[:5])
assert not errors, 'FPGA calculation disagrees with ARM' 

## Student questions
1. Which operations execute on ARM, which in the AXI GPIO blocks, and which in the `mac8` RTL?
2. Why must `.bit` and `.hwh` have the same base name?
3. Why do we configure the result register as GPIO **input**?
4. What is the maximum mathematical value of `Y` for the selected widths?